In [ ]:
from napari import Viewer

import sys
from faim_ipa.utils import get_git_root
from pathlib import Path
from ipyfilechooser import FileChooser
from ipywidgets import widgets
from IPython.display import display
import dask
import numpy as np

sys.path.append(str(get_git_root()))

from source.s01_convert_to_zarr.create_selection_csv_utils import (
    get_experiment_widget,
    load_positions,
)

viewer = Viewer()

# Choose the git-repo on the file-server

In [ ]:
fc = FileChooser(
    path=get_git_root(),
    layout=widgets.Layout(width="100%"),
    show_only_dirs=True,
    title="Select Processing Directory",
)
display(fc)

In [ ]:
root_dir = Path(fc.selected)

# Select the experiment 

In [ ]:
exp = get_experiment_widget(root_dir)
display(exp)

In [ ]:
config, position_paths = load_positions(exp, root_dir, from_input_dir=False)

In [ ]:
pos = widgets.Dropdown(
    options=[(p.name, p) for p in position_paths],
    description="Position:",
    disabled=False,
)

# Select position and visualize

In [ ]:
pos

In [ ]:
da_img = dask.array.from_zarr(pos.value, component="0")

viewer.layers.clear()
viewer.add_image(
    da_img[:, 0],
    contrast_limits=np.quantile(da_img[0, 0].compute(), (0, 1)),
    multiscale=False,
    name=f"{pos.value.name} - Channel 1",
)
viewer.add_image(
    da_img[:, 1],
    contrast_limits=np.quantile(
        da_img[da_img.shape[0] // 2, 1].compute().max(0), (0.5, 0.998)
    ),
    colormap="green",
    blending="additive",
    multiscale=False,
    name=f"{pos.value.name} - Channel 2",
)
viewer.dims.set_current_step(0, 0)